In [67]:
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)          # don't wrap at 80 chars  
pd.set_option('display.max_colwidth', None)  

In [68]:
import pandas as pd  
import glob  
import os  
import re

folder_path = 'L:/2026 Pilar Plant Files/Position Control/'

# Filter to only LENOX files (case-insensitive) and exclude temp lock files (~$)  
all_excel_files = glob.glob(os.path.join(folder_path, '*.xlsx'))  
lenox_files = [  
    f for f in all_excel_files   
    if 'LENOX' in os.path.basename(f).upper()   
]

all_data = {}

for file in lenox_files:  
    file_name = os.path.basename(file)  
    try:  
        df = pd.read_excel(file, sheet_name='PositionDetail', skiprows=12)  
        df['source_file'] = file_name  
          
        # Extract date (format YYYY-MM-DD) from the end of the filename  
        match = re.search(r'(\d{4}-\d{2}-\d{2})', file_name)  
        if match:  
            df['file_date'] = pd.to_datetime(match.group(1))  
        else:  
            df['file_date'] = pd.NaT  
          
        all_data[file_name] = df  
        print(f"Loaded: {file_name} ({len(df)} rows)")  
    except Exception as e:  
        print(f"Skipped {file_name} — Error: {e}")

# Combine into one master DataFrame  
combined_df = pd.concat(all_data.values(), ignore_index=True)  
print(f"\nTotal combined rows: {len(combined_df)}")  

Loaded: LENOX POSNEXEC LENOX_EXEC 2026-01-03.xlsx (5180 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-01-17.xlsx (5177 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-01-24.xlsx (5183 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-01-31.xlsx (5187 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-02-07.xlsx (5188 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-02-14.xlsx (5187 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-02-21.xlsx (5188 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-02-28.xlsx (5199 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-03-07.xlsx (5196 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-03-21.xlsx (5209 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-04-04.xlsx (5140 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-04-18.xlsx (5145 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-05-16.xlsx (5158 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-05-23.xlsx (5147 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-06-06.xlsx (5153 rows)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-06-13.xlsx (5155 rows)
Loaded: 

In [73]:

pos_cont = combined_df[['file_date', 'Rpt Dept', 'Rpt Dept Desc', 'Job Code', 'Jobcode Title', 'Posn Number', 'Posn Status',
                         'Filled/ Open', 'Incumbent Name', 'Emplid', 'Incumbent Status']]

pc_no_null = pos_cont[(pos_cont['Rpt Dept'].notna()) & (~pos_cont['Posn Number'].isin(['Per Diem', 'Flexstaff']))]

roster = pd.read_excel("C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/roster_budget.xlsx", skiprows=6)

roster_dates = pd.merge(pc_no_null[['file_date']].drop_duplicates(), roster, how='cross')

roster_dates['Position #'] = pd.to_numeric(roster_dates['Position #'], errors='coerce')
pc_no_null['Posn Number'] = pd.to_numeric(pc_no_null['Posn Number'], errors='coerce')


roster_positions = pd.merge(pc_no_null, roster_dates, how='outer', left_on=['file_date', 'Posn Number'], right_on=['file_date', 'Position #'])

roster_positions['position num'] = roster_positions['Position #'].fillna(roster_positions['Posn Number'])

roster_comp = roster_positions[['file_date', 'position num','2026 Budget', '2027 Budget','Posn Number', 'Posn Status', 'Filled/ Open', 'Incumbent Name', 'Emplid', 'Position #', 
                                'Status', 'Employee ID', 'Employee Name', 'Rpt Dept', 'Rpt Dept Desc', 'Job Code_x', 'Jobcode Title', 'GL Dept ID', 'GL Dept Desc', 'Job Code_y', 
                                'Job Code Desc', 'VP', 'Director']]

roster_comp_na = roster_comp[(roster_comp['position num'].notna()) & (roster_comp['2026 Budget'] != 0) &
                             (roster_comp['2026 Budget'].notna())]


In [78]:
full_roster = roster_comp_na[['file_date', 'VP', 'Director', 'GL Dept ID', 'GL Dept Desc', 'Job Code_y', 'Job Code Desc', 'Position #',  'Employee ID', 
                              'Employee Name', 'Filled/ Open', '2026 Budget', '2027 Budget']].rename(columns={'file_date': 'As of Date', 'Job Code_y': 'Job Code'})

full_roster['Filled/ Open'] = full_roster['Filled/ Open'].fillna('Open')


total_positions = full_roster.groupby(['As of Date'])[['Position #']].nunique().reset_index()
open_positions = (full_roster[full_roster['Filled/ Open'] == 'Open'].groupby('As of Date')['Position #'].nunique().reset_index(name='unfilled_count'))  

dated_open = pd.merge(total_positions, open_positions, how='outer', on='As of Date')
dated_open['vacancy'] = (dated_open['unfilled_count'] / dated_open['Position #'])

display(full_roster)
display(dated_open)

with pd.ExcelWriter('vacancy_report.xlsx', engine='openpyxl') as writer:  
    dated_open.to_excel(writer, sheet_name='Vacancy by Date', index=False)  
    full_roster.to_excel(writer, sheet_name='Roster Detail', index=False)  

,As of Date,VP,Director,GL Dept ID,GL Dept Desc,Job Code,Job Code Desc,Position #,Employee ID,Employee Name,Filled/ Open,2026 Budget,2027 Budget
0,2026-01-03,"Browne, Kevin","Glennon, Peter",15655000.0,Dialysis Acute Dialysis,904196.0,Clinical Support Services Assistant,10010100.0,188584,"Murphy,Lawrence J",Filled,0.88,0.88
3,2026-01-03,"Hathaway, Taylor A","Mohabir, Amanda",15653005.0,Radiation Oncology,203264.0,"Senior Manager, IT&S Program Management",10016015.0,380533,"Morgenstern,Jason",Filled,0.20,0.20
8,2026-01-03,"Browne, Kevin","Glennon, Peter",15655000.0,Dialysis Acute Dialysis,201464.0,Registered Nurse,10041060.0,178823,"Varilla,John Ernest L",Filled,1.00,1.00
9,2026-01-03,"Hathaway, Taylor A","Ross, Tena",15621030.0,Psych - IP Psych,102920.0,PIC (LJ),10044361.0,122572,"Braga,Raphael",Filled,0.16,0.16
11,2026-01-03,"Jurik, Christopher","Debacker, Johanna M",15600000.0,Hospital Administration,903315.0,Executive Assistant,10047370.0,409193,"Vil,Nathalie G.",Filled,1.00,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
237670,2026-09-26,"Leggio, Joe","Kubba, Hala",15601325.0,Medical Tourism,201551.0,Senior Executive Assistant,81099605.0,607110,"Hampton,Chelaine",Filled,1.00,1.00
237676,2026-09-26,"Browne, Kevin","Bates, Eric M",15613047.0,NRSG - 7 East MICU,108102.0,RN,81099744.0,NaN,NaN,Open,1.00,1.00
237685,2026-09-26,"Cole, Brandon L","Grinberg, Stan",15650170.0,Pathology - Clinical,108324.0,Phlebotomist,81100719.0,NaN,NaN,Open,0.50,0.50
237691,2026-09-26,"Jurik, Christopher","Debacker, Johanna M",15600000.0,Hospital Administration,903742.0,Finance & Operations Management Associate,81101190.0,586527,"Cohen,Ava Elizabeth",Filled,1.00,1.00


,As of Date,Position #,unfilled_count,vacancy
0,2026-01-03,3536,279,0.078903
1,2026-01-17,3536,271,0.076640
2,2026-01-24,3536,273,0.077206
3,2026-01-31,3536,262,0.074095
4,2026-02-07,3536,263,0.074378
5,2026-02-14,3536,255,0.072115
6,2026-02-21,3536,251,0.070984
7,2026-02-28,3536,240,0.067873
8,2026-03-07,3536,248,0.070136
9,2026-03-21,3536,247,0.069853
